<a href="https://colab.research.google.com/github/KingsWorkflow/KtmAir-Quality-and-Weather-ETL-Pipeline/blob/main/DataAnalaysisFInalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kathmandu Valley Air Quality and Weather ETL Pipeline: Understanding Pollution Patterns for Public Health Awareness**

**Setup**




In [2]:
import json, time, hashlib, requests
import pandas as pd
from pathlib import Path
from datetime import datetime

PROJECT = Path("/content/kathmandu_aq_etl")
RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT)

Project folder: /content/kathmandu_aq_etl


**API configuration and a safe request helper**

In [3]:
from google.colab import userdata
import requests
import time

API_KEY = userdata.get('OPENAQ')
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

def api_get(path, params=None, retries=5):
    """GET with retry/backoff for rate limits, timeouts, and temporary server errors."""
    url = f"{BASE_URL}{path}"
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (408, 429, 500, 502, 503, 504):
                wait = 20 * (attempt + 1)
                print(f"  HTTP {r.status_code} for {path}, retrying in {wait}s (attempt {attempt + 1}/{retries})...")
                time.sleep(wait)
                continue
            r.raise_for_status()
        except (requests.exceptions.RequestException, RuntimeError) as e:
            if attempt == retries - 1:
                raise
            wait = 20 * (attempt + 1)
            print(f"  Request exception: {e}, retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {retries} tries: {url}")

# Quick connectivity test
print(api_get("/countries", {"limit": 1})["meta"])

{'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 1, 'found': 159}


**Discover monitoring stations in the Kathmandu Valley**

In [4]:
# Bounding box: min_lon, min_lat, max_lon, max_lat (covers Kathmandu, Lalitpur, Bhaktapur)
KTM_BBOX = "85.15,27.55,85.60,27.85"

resp = api_get("/locations", {"bbox": KTM_BBOX, "limit": 1000})
locations = resp["results"]
print("Stations found:", len(locations))

# Preserve the raw API response exactly as received
with open(RAW / "openaq_locations_raw.json", "w") as f:
    json.dump(resp, f, indent=2)

station_rows, sensor_rows = [], []
for loc in locations:
    coords = loc.get("coordinates") or {}
    station_rows.append({
        "location_id": loc["id"],
        "name": loc.get("name"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
        "provider": (loc.get("provider") or {}).get("name"),
        "owner": (loc.get("owner") or {}).get("name"),
        "is_mobile": loc.get("isMobile"),
        "first_utc": (loc.get("datetimeFirst") or {}).get("utc"),
        "last_utc": (loc.get("datetimeLast") or {}).get("utc"),
    })
    for s in loc.get("sensors", []):
        p = s.get("parameter") or {}
        sensor_rows.append({
            "sensor_id": s["id"],
            "location_id": loc["id"],
            "parameter": p.get("name"),
            "units": p.get("units"),
        })

stations_raw = pd.DataFrame(station_rows)
sensors_raw = pd.DataFrame(sensor_rows)
stations_raw.to_csv(RAW / "stations_raw.csv", index=False)
sensors_raw.to_csv(RAW / "sensors_raw.csv", index=False)

display(stations_raw)
display(sensors_raw.groupby(["location_id", "parameter"]).size().unstack(fill_value=0))

Stations found: 60


,location_id,name,latitude,longitude,provider,owner,is_mobile,first_utc,last_utc
0,3459,Embassy Kathmandu,27.738703,85.336206,AirNow,Unknown Governmental Organization,False,2017-03-03T00:00:00Z,2026-06-17T23:15:00Z
1,3460,Phora Durbar Kathman,27.712464,85.315703,AirNow,Unknown Governmental Organization,False,2017-03-03T00:00:00Z,2025-03-24T14:15:00Z
2,1236017,"Dhathutole, Handigaun",27.727502,85.330135,AirGradient,Unknown Governmental Organization,False,2023-07-15T13:00:00Z,2026-06-01T04:00:00Z
3,2848864,"Dabali, Handigaun",27.681891,85.287078,AirGradient,Unknown Person,False,2024-05-16T03:00:00Z,2026-10-09T10:00:00Z
4,4121681,Kathmandu Institute of Applied Sciences (KIAS),27.671693,85.301670,AirGradient,Sapana Bhujel,False,2025-05-01T09:00:00Z,2026-10-09T10:00:00Z
5,5506835,Gaushala Chowk (SC-01) - GD Labs,27.707763,85.343189,AirGradient,Green Decision Labs and Research,False,2025-09-19T00:00:00Z,2026-10-09T10:00:00Z
6,5509787,Baluwatar (SC-02) - GD Labs,27.724944,85.331062,AirGradient,Green Decision Labs and Research,False,2025-09-25T07:00:00Z,2026-10-09T10:00:00Z
7,5509788,Lagankhel (SC - 05) - GD Labs,27.666459,85.323093,AirGradient,Green Decision Labs and Research,False,2025-10-03T00:00:00Z,2026-10-09T10:00:00Z
8,5518333,Sifal(SC-03)- GD Labs,27.710794,85.340591,AirGradient,Green Decision Labs and Research,False,2025-12-24T08:00:00Z,2026-10-09T10:00:00Z
9,5519358,Lamtangil (SC-04)- GD Labs,27.731368,85.336783,AirGradient,Green Decision Labs and Research,False,2025-12-24T09:00:00Z,2026-10-09T10:00:00Z


parameter,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
location_id,,,,,,,
3459,1,0,0,1,0,0,0
3460,1,0,0,1,0,0,0
1236017,0,1,1,1,1,1,1
2848864,0,1,1,1,1,1,1
4121681,0,1,0,1,1,1,1
5506835,0,1,0,1,1,1,1
5509787,0,1,0,1,1,1,1
5509788,0,1,0,1,1,1,1
5518333,0,1,0,1,1,1,1


**`Choosing sensors (pollutants, active since 2023)`**

In [5]:
# Keep only stations that have a start and end date (station 59 has None)
stations_ok = stations_raw.dropna(subset=["first_utc", "last_utc"])

stations_ok = stations_ok[stations_ok["last_utc"] >= "2023-01-01"]

# Keep only the pollutants we want
wanted = ["pm25", "pm10", "o3"]
selected = sensors_raw[sensors_raw["parameter"].isin(wanted)]

# Keep only sensors that belong to the good stations
selected = selected[selected["location_id"].isin(stations_ok["location_id"])]

print(len(selected), "sensors selected")
print(selected["parameter"].value_counts())

63 sensors selected
parameter
pm25    59
o3       2
pm10     2
Name: count, dtype: int64


**Testing with ONE sensor first**

In [6]:
# Sensor 7713 is the o3 sensor from your earlier output
test = api_get("/sensors/7713/days", {
    "datetime_from": "2025-01-01",
    "datetime_to": "2025-02-01",
    "limit": 1000
})

print("Rows returned:", len(test["results"]))
print(test["results"][0])

Rows returned: 1000
{'value': 0.029, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, 'period': {'label': '1day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2017-03-02T18:15:00Z', 'local': '2017-03-03T00:00:00+05:45'}, 'datetimeTo': {'utc': '2017-03-03T18:15:00Z', 'local': '2017-03-04T00:00:00+05:45'}}, 'coordinates': None, 'summary': {'min': 0.003000000026077032, 'q02': 0.0033599999733269215, 'q25': 0.006500000134110451, 'median': 0.032999999821186066, 'q75': 0.05000000074505806, 'q98': 0.05764000117778778, 'max': 0.057999998331069946, 'avg': 0.028999999165534973, 'sd': 0.021919043734669685}, 'coverage': {'expectedCount': 24, 'expectedInterval': '24:00:00', 'observedCount': 19, 'observedInterval': '19:00:00', 'percentComplete': 79.0, 'percentCoverage': 79.0, 'datetimeFrom': {'utc': '2017-03-03T00:00:00Z', 'local': '2017-03-03T05:45:00+05:45'}, 'datetimeTo': {'utc': '2017-03-03T18:00:00Z', 'local': '2017-03-03T23:45:00+0

**Full Pull**

In [13]:
all_rows = []

for i, row in selected.iterrows():
    sensor_id = int(row["sensor_id"])
    page = 1

    while True:
        data = api_get("/sensors/" + str(sensor_id) + "/days",
                       {"limit": 1000, "page": page})
        results = data["results"]

        for r in results:
            summary = r.get("summary") or {}
            coverage = r.get("coverage") or {}

            all_rows.append({
                "sensor_id": sensor_id,
                "location_id": row["location_id"],
                "parameter": row["parameter"],
                "units": row["units"],
                "date_local": r["period"]["datetimeFrom"]["local"],
                "avg_value": r["value"],
                "min_value": summary.get("min"),
                "max_value": summary.get("max"),
                "percent_complete": coverage.get("percentComplete"),
            })

        if len(results) < 1000:      # last page reached
            break
        page = page + 1
        time.sleep(1.1)

    time.sleep(1.1)
    print("done sensor", sensor_id, "| total rows so far:", len(all_rows))

readings_raw = pd.DataFrame(all_rows)
readings_raw.to_csv(RAW / "readings_daily_raw.csv", index=False)   # overwrites the bad file

print(readings_raw.shape)
print("Duplicate rows:", readings_raw.duplicated().sum())
print("Date range:", readings_raw["date_local"].min(), "to", readings_raw["date_local"].max())

done sensor 7713 | total rows so far: 2320
done sensor 7710 | total rows so far: 5396
done sensor 7712 | total rows so far: 8048
done sensor 7711 | total rows so far: 10871
done sensor 8462426 | total rows so far: 11111
done sensor 6530219 | total rows so far: 11537
done sensor 9235741 | total rows so far: 11776
done sensor 9235746 | total rows so far: 12502
done sensor 12914045 | total rows so far: 12923
done sensor 14153358 | total rows so far: 13211
done sensor 14207425 | total rows so far: 13415
done sensor 14272754 | total rows so far: 13748
done sensor 14975414 | total rows so far: 13947
done sensor 14976381 | total rows so far: 14166
done sensor 14153468 | total rows so far: 14265
done sensor 14153373 | total rows so far: 14343
done sensor 14153363 | total rows so far: 14664
done sensor 14153368 | total rows so far: 15015
done sensor 14153378 | total rows so far: 15240
done sensor 14153463 | total rows so far: 15312
done sensor 14573141 | total rows so far: 15440
done sensor 141

In [14]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 27.7172,          # Kathmandu
    "longitude": 85.3240,
    "start_date": "2023-01-01",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,surface_pressure",
    "timezone": "Asia/Kathmandu"
}

response = requests.get(url, params=params, timeout=120)
print("Status code:", response.status_code)

weather_json = response.json()

# Save the original response first (raw data stays untouched)
with open(RAW / "weather_raw.json", "w") as f:
    json.dump(weather_json, f)

# Convert to a DataFrame and save as CSV
weather_raw = pd.DataFrame(weather_json["hourly"])
weather_raw.to_csv(RAW / "weather_raw.csv", index=False)

print(weather_raw.shape)
weather_raw.head()

Status code: 200
(26304, 6)


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,surface_pressure
0,2023-01-01T00:00,7.9,88,0.0,1.1,872.6
1,2023-01-01T01:00,8.0,85,0.0,2.9,872.3
2,2023-01-01T02:00,8.0,81,0.0,4.1,871.9
3,2023-01-01T03:00,8.2,76,0.0,3.6,871.8
4,2023-01-01T04:00,8.3,72,0.0,2.3,871.8


**Hourly weather from Open-Meteo (free, no key)**

In [16]:
import os

print("Files in raw folder:")
print(os.listdir(RAW))
print()
print("Daily readings:", readings_raw.shape)
print("Weather rows:  ", weather_raw.shape)
print("Stations:      ", stations_raw.shape)
print("Sensors:       ", sensors_raw.shape)

# The project needs at least a few hundred rows
assert len(readings_raw) > 500, "Not enough rows!"
print("Phase 1 complete")

Files in raw folder:
['stations_raw.csv', 'weather_raw.json', 'weather_raw.csv', 'openaq_locations_raw.json', 'readings_daily_raw.csv', 'sensors_raw.csv']

Daily readings: (23189, 9)
Weather rows:   (26304, 6)
Stations:       (60, 9)
Sensors:        (292, 4)
Phase 1 complete


In [17]:
print("Total rows:", len(readings_raw))
print("Duplicate rows:", readings_raw.duplicated().sum())
print("Earliest date:", readings_raw["date_local"].min())
print("Latest date:  ", readings_raw["date_local"].max())

Total rows: 23189
Duplicate rows: 0
Earliest date: 2017-03-03T00:00:00+05:45
Latest date:   2026-10-08T00:00:00+05:45
